# Gradient-boosted trees for hourly PM2.5 (London, 2024)

This notebook specifies an **XGBoost** regressor on a **tabular** representation of the hourly air-quality panel. Unlike recurrent architectures, tree ensembles do not ingest variable-length sequences directly; instead, calendar structure and explicit **lagged** concentrations encode temporal dependence, which permits transparent comparison of lag importance against the deep-learning baseline.

## Imports and project paths

Paths resolve whether the notebook kernel starts in the repository root or in `notebooks`, matching the convention used in earlier modelling notebooks.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor


def project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "london_2024.csv").exists():
        return cwd
    if (cwd.parent / "data" / "london_2024.csv").exists():
        return cwd.parent
    raise FileNotFoundError(
        "Could not find data/london_2024.csv. "
        "Run with cwd = thesis_2026 or thesis_2026/notebooks."
    )


PROJECT_ROOT = project_root()
DATA_PATH = PROJECT_ROOT / "data" / "london_2024.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "xgboost_model.joblib"

In [ ]:
# Reproducibility: library versions and RNG seeds (copy output into thesis appendix if needed)
import random
import sys

RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

try:
    import tensorflow as tf

    tf.random.set_seed(RNG_SEED)
except ImportError:
    pass

print("Python:", sys.version.split()[0])
for _name, _mod in (
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("sklearn", "sklearn"),
    ("xgboost", "xgboost"),
    ("matplotlib", "matplotlib"),
):
    try:
        _m = __import__(_mod)
        print(f"{_name}:", getattr(_m, "__version__", "?"))
    except ImportError:
        print(f"{_name}: (not installed)")


## Raw panel preparation

Static geographic fields are removed. Observations are aligned to a strict hourly index so that each row corresponds to one clock hour; forward-fill carries the last measured multivariate state forward across short gaps, which avoids fabricating variation while keeping the timeline continuous for lag construction.

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["time"])
df = df.drop(columns=["city", "latitude", "longitude"])
df = df.sort_values("time").set_index("time")
df = df.asfreq("h")
df = df.ffill()

## From series to supervised table: feature engineering philosophy

Gradient boosting assumes **conditionally independent rows** with a fixed feature vector. To respect the underlying stochastic process while staying within that framework, we **discretise calendar time** (hour, day-of-week, month) to capture systematic seasonality and weekly routines, and we **copy past realisations** of `pm2_5` as separate columns (`t-1`, `t-2`, `t-24`). The one-day lag (`t-24`) proxies persistence at the same clock hour on the previous day—a coarse analogue of diurnal memory—whereas short lags (`t-1`, `t-2`) encode local inertia. Rows with insufficient history for the largest lag are dropped rather than imputed, so that every training example is defined solely from **strictly past** PM2.5 values relative to the target hour. Contemporaneous co-pollutant readings remain as additional regressors for the same timestamp, which corresponds to a **nowcasting** interpretation (all inputs are realised in the same hour as the target); this should be stated explicitly when contrasting against purely autoregressive horizons.

In [ ]:
eng = df.copy()
idx = eng.index
eng["hour"] = idx.hour.astype(np.int32)
eng["day_of_week"] = idx.dayofweek.astype(np.int32)
eng["month"] = idx.month.astype(np.int32)

eng["pm2_5_lag1"] = eng["pm2_5"].shift(1)
eng["pm2_5_lag2"] = eng["pm2_5"].shift(2)
eng["pm2_5_lag24"] = eng["pm2_5"].shift(24)

eng = eng.dropna()
print(f"Rows after lagging and dropna: {len(eng)}")

## Chronological train–test partition

The engineered table is split **in time order** (80% earliest rows for training, 20% most recent for testing) without shuffling. The design matrix **excludes** the contemporaneous `pm2_5` column so that the target is not duplicated as an input; lagged PM2.5 columns remain legitimate predictors.

In [ ]:
split = int(len(eng) * 0.8)
train_df = eng.iloc[:split]
test_df = eng.iloc[split:]

y_train = train_df["pm2_5"].astype(float)
y_test = test_df["pm2_5"].astype(float)
X_train = train_df.drop(columns=["pm2_5"])
X_test = test_df.drop(columns=["pm2_5"])

print(f"Train: {len(X_train)} rows, {X_train.shape[1]} features")
print(f"Test:  {len(X_test)} rows")
print(f"Train window: {train_df.index.min()} → {train_df.index.max()}")
print(f"Test window:  {test_df.index.min()} → {test_df.index.max()}")

## Model estimation

XGBoost fits an additive ensemble of weak learners with regularised splitting; `random_state` fixes subsampling and any stochastic tie-breaking so that thesis results are reproducible across runs on the same machine.

In [ ]:
model = XGBRegressor(random_state=42)
model.fit(X_train, y_train)


## Out-of-sample error and interpretable importance

Root mean squared error and mean absolute error summarise scale-dependent fit on the held-out tail. **Gain-based** importance from XGBoost aggregates the improvement in the loss attributable to each feature across splits; it supports narrative interpretation of which calendar indicators and which memory horizons the ensemble exploits most heavily, subject to the caveat that correlated lags can share credit.

In [ ]:
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = float(np.sqrt(mse))
mae = float(mean_absolute_error(y_test, y_pred))
print(f"Test RMSE (µg/m³): {rmse:.4f}")
print(f"Test MAE  (µg/m³): {mae:.4f}")

fig, ax = plt.subplots(figsize=(8, 6))
xgb.plot_importance(model, ax=ax, max_num_features=20, importance_type="gain")
ax.set_title("XGBoost feature importance (gain)")
plt.tight_layout()
plt.show()

## Saving model metadata

A companion JSON file records the training timestamp, dataset version, feature list, and evaluation metrics alongside the model artifact.

In [ ]:
import json
from datetime import datetime, timezone

metadata = {
    "model": "xgboost",
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "data_version": str(DATA_PATH.name),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "feature_columns": list(X_train.columns),
    "metrics": {"rmse": round(rmse, 4), "mae": round(mae, 4)},
    "notes": "XGBRegressor with calendar + lag features; 80/20 chronological split.",
}
meta_path = MODEL_PATH.parent / "xgboost_metadata.json"
with meta_path.open("w") as fh:
    json.dump(metadata, fh, indent=2)
print(f"Saved metadata to {meta_path}")
print(json.dumps(metadata, indent=2))